In [1]:
#cell 1
# Setup imports

import json
import re
import string
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell 2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
#cell 3
# Define file paths and model names

BASE_DIR = Path("/content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa")

MODEL_FILES = {
    "qwen3.5": BASE_DIR / "2wikimultihopqa_answer_qwen3.5.json",
    "gemma4": BASE_DIR / "2wikimultihopqa_answer_gemma4.json",
    "gpt-oss": BASE_DIR / "2wikimultihopqa_answer_gpt_oss.json",
}

In [4]:
#cell 4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    This follows the set-overlap formula in the image.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell 5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell 6
# Load one model file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_model(model_name, file_path):
    """
    Evaluate all questions for one model.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "model": model_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell 7
# Evaluate all models

def evaluate_all_models(model_files):
    """
    Evaluate every model file and combine results.
    """
    all_dfs = []

    for model_name, file_path in model_files.items():
        model_df = evaluate_model(model_name, file_path)
        all_dfs.append(model_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
#cell 8
# Build overall and per-type summaries

def build_summary(scores_df):
    """
    Create overall and per-type macro F1 summaries.
    """
    overall_df = (
        scores_df
        .groupby("model", as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df.insert(1, "type", "overall")

    type_df = (
        scores_df
        .groupby(["model", "type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    model_order = ["qwen3.5", "gemma4", "gpt-oss"]
    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["model", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
#cell9
# Final output only

scores_df = evaluate_all_models(MODEL_FILES)
summary_df = build_summary(scores_df)

display(summary_df)

,model,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,qwen3.5,overall,1000,0.802874,0.862444,0.816572,81.657245
1,qwen3.5,bridge_comparison,250,0.950233,0.969333,0.952775,95.277460
2,qwen3.5,comparison,250,0.925000,0.960000,0.933362,93.336190
3,qwen3.5,compositional,250,0.597638,0.735200,0.633902,63.390194
4,qwen3.5,inference,250,0.738624,0.785243,0.746251,74.625135
5,gemma4,overall,1000,0.842921,0.907376,0.856725,85.672521
6,gemma4,bridge_comparison,250,0.971038,0.984000,0.973076,97.307619
7,gemma4,comparison,250,0.951505,0.981000,0.956346,95.634632
8,gemma4,compositional,250,0.599173,0.750057,0.635511,63.551105
9,gemma4,inference,250,0.849969,0.914448,0.861967,86.196728
